# 06 — Predictive Analysis (Classification)

Trains and compares Logistic Regression, SVM, Decision Tree, and KNN
classifiers to predict Falcon 9 first-stage landing success. Runs fully
offline against `falcon9_dataset.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

df = pd.read_csv("../falcon9_dataset.csv")
X = pd.get_dummies(df[["LaunchSite","Orbit","PayloadMass","GridFins","Reused","Legs","Block","FlightNumber"]],
                    columns=["LaunchSite","Orbit"])
y = df["Class"]


### Train/test split and scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=7, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print(f"Train: {X_train.shape[0]}  Test: {X_test.shape[0]}")


### Grid search over four classifiers

In [ ]:
grids = {
    "Logistic Regression": (LogisticRegression(max_iter=1000), {"C":[0.01,0.1,1,10]}),
    "SVM": (SVC(), {"C":[0.1,1,10], "kernel":["linear","rbf"]}),
    "Decision Tree": (DecisionTreeClassifier(random_state=7), {"max_depth":[2,3,4,5,None]}),
    "KNN": (KNeighborsClassifier(), {"n_neighbors":[3,5,7,9]}),
}

results = {}
best_models = {}
for name, (est, params) in grids.items():
    gs = GridSearchCV(est, params, cv=5)
    gs.fit(X_train_s, y_train)
    pred = gs.predict(X_test_s)
    acc = accuracy_score(y_test, pred)
    results[name] = {"cv_best": gs.best_score_, "test_acc": acc, "best_params": gs.best_params_}
    best_models[name] = gs.best_estimator_
    print(f"{name:22s} CV={gs.best_score_:.3f}  Test={acc:.3f}  params={gs.best_params_}")


### Select the best model

With a small held-out test set, ties on test accuracy are expected. Break
the tie using cross-validated accuracy, which uses far more of the data.

In [ ]:
best_name = max(results, key=lambda k: (results[k]["test_acc"], results[k]["cv_best"]))
print("Best model:", best_name, results[best_name])


### Accuracy comparison chart

In [ ]:
names = list(results.keys())
accs = [results[n]["test_acc"] * 100 for n in names]
fig, ax = plt.subplots(figsize=(6,4))
ax.bar(names, accs, color="#14b8a6")
ax.set_ylabel("Test accuracy (%)"); ax.set_title("Classification Accuracy on Test Set")
plt.xticks(rotation=15)
plt.show()


### Confusion matrix for the best model

In [ ]:
pred_best = best_models[best_name].predict(X_test_s)
cm = confusion_matrix(y_test, pred_best)
print(cm)

fig, ax = plt.subplots(figsize=(4,4))
im = ax.imshow(cm, cmap="viridis")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i,j]), ha="center", va="center", color="white", fontsize=16)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title(f"{best_name} Confusion Matrix")
plt.show()
